# GNN-RAG for animal image editing

This notebook compares plain Stable Diffusion, CLIP-based retrieval, and graph-based retrieval for prompts that add attributes to animal images. It uses AwA2 as an image and attribute memory, and evaluates generated images with CLIP and optional human ratings.

The graph passes AwA2 attribute information between animal classes and retrieved images without training model weights. A prompt parser identifies the animal and requested attributes; retrieval then selects an image and sets img2img strength from the edits still needed. Unknown edits receive the highest strength.

The graph covers only AwA2’s 85 attributes. Stable Diffusion 1.5 may not follow requests such as an exact number of limbs, so those prompts are illustrative rather than benchmark items.


### 1. Install

In [ ]:
# torchao is removed because Colab's version breaks recent diffusers (FqnToConfig error).
# If diffusers was already imported in this session, restart the session after this cell.
!pip -q uninstall -y torchao
!pip -q install diffusers remotezip gradio

### 2. Configuration

In [ ]:
import os, io, gc, re, json, time, random, math, threading
import concurrent.futures as cf
from collections import defaultdict
import numpy as np
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from PIL import Image

SEED           = 42
N_PER_CLASS    = 40                   # same memory as the previous notebook (cache is reused)
N_NOMINAL      = 25                   # nominal denoising schedule for every pipeline
FIXED_STRENGTH = 0.8                  # normal RAG: 20 of 25 steps
STRENGTH = {"none": 0.5, "color": 0.6, "scene": 0.7, "pattern": 0.75, "size": 0.75,
            "species": 0.8, "part": 0.85, "unknown": 0.9}     # initial guesses; replaced by calibration
INITIAL_STRENGTH = dict(STRENGTH)
ALPHA_CLIP     = 0.1                  # weight of CLIP similarity inside GNN retrieval
# Message passing: how much a single photo's own CLIP evidence counts vs its animal's profile.
# Colour and background vary from photo to photo; body parts and patterns belong to the species.
MP_BETA        = {"color": 0.3, "scene": 0.6, "pattern": 0.2, "size": 0.2, "part": 0.2}
N_PER_CATEGORY = 20                   # benchmark prompts per edit category (attribute-only: up to 10)
N_SEEDS        = 2                    # seeds per benchmark prompt
N_CAL          = 8                    # calibration prompts per edit category (disjoint from benchmark)
CAL_STRENGTHS  = (0.4, 0.5, 0.6, 0.7, 0.8, 0.9)
CAL_TOL        = 0.5                  # calibrated = lowest strength within 0.5 CLIP points of plain_25
RECALIBRATE    = False                # True forces calibration to run again
USE_TURBO      = True                 # SD-Turbo baselines (extra ~3.5 GB download)
TURBO_ID       = "stabilityai/sd-turbo"
HUMAN_N        = 60                   # (prompt, seed) pairs shown to human raters per comparison
GUIDANCE       = 7.5
SD_ID          = "stable-diffusion-v1-5/stable-diffusion-v1-5"
CLIP_RETRIEVE  = "openai/clip-vit-base-patch32"
CLIP_JUDGE     = "openai/clip-vit-large-patch14"
AWA_DATA       = "https://cvml.ista.ac.at/AwA2/AwA2-data.zip"
AWA_BASE       = "https://cvml.ista.ac.at/AwA2/AwA2-base.zip"
UNSEEN = {"chimpanzee", "giant+panda", "leopard", "persian+cat", "pig", "hippopotamus",
          "humpback+whale", "raccoon", "rat", "seal"}
COLORS = ["black", "white", "blue", "brown", "gray", "orange", "red", "yellow"]

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT = "./graph_rag_image_diffusion"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/graph_rag_image_diffusion"
except Exception:
    pass
assert DEVICE == "cuda", ("No GPU in this session. Runtime -> Change runtime type -> T4 GPU, then rerun "
                          "from the top. If no GPU is offered, the free-tier GPU quota is used up for now.")
assert OUT.startswith("/content/drive"), \
    "Google Drive is not mounted — results would be lost on disconnect. Mount it and rerun."
IMG_DIR, GEN_DIR = os.path.join(OUT, "memory"), os.path.join(OUT, "generated_v2")
os.makedirs(IMG_DIR, exist_ok=True); os.makedirs(GEN_DIR, exist_ok=True)
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "", "| ->", OUT)

### 3. AwA2 attribute table + image memory (reuses the previous notebook's cache on Drive)

In [ ]:
def open_zip(url):
    try:
        from remotezip import RemoteZip
        z = RemoteZip(url); z.namelist()
        return z, "remote"
    except Exception as err:
        import urllib.request, zipfile
        local = os.path.join("/content" if os.path.isdir("/content") else ".", os.path.basename(url))
        if not os.path.exists(local):
            print(f"range requests unavailable ({err}); downloading {url} in full ...")
            urllib.request.urlretrieve(url, local)
        return zipfile.ZipFile(local), "local"

meta_path = os.path.join(OUT, "awa2_meta.json")
if os.path.exists(meta_path):
    META = json.load(open(meta_path))
else:
    z, mode = open_zip(AWA_DATA)
    names = z.namelist()
    want = {"classes.txt", "predicates.txt", "predicate-matrix-continuous.txt", "predicate-matrix-binary.txt"}
    found = {os.path.basename(n): n for n in names if os.path.basename(n) in want}
    zb = z if want <= set(found) else open_zip(AWA_BASE)[0]
    if zb is not z:
        found = {os.path.basename(n): n for n in zb.namelist() if os.path.basename(n) in want}
    rd = lambda f: zb.read(found[f]).decode().strip().splitlines()
    META = dict(classes=[l.split()[-1] for l in rd("classes.txt")],
                predicates=[l.split()[-1] for l in rd("predicates.txt")],
                cont=[[float(v) for v in l.split()] for l in rd("predicate-matrix-continuous.txt")],
                binary=[[int(v) for v in l.split()] for l in rd("predicate-matrix-binary.txt")],
                images={})
    for n in names:
        parts = n.split("/")
        if len(parts) >= 3 and parts[-3] == "JPEGImages" and n.lower().endswith(".jpg"):
            META["images"].setdefault(parts[-2], []).append(n)
    json.dump(META, open(meta_path, "w"))

CLASSES, PREDS = META["classes"], META["predicates"]
CONT, BINARY = np.array(META["cont"]), np.array(META["binary"])
ATTR = np.clip(CONT, 0, 100) / 100.0                                  # animal x attribute in [0, 1]
cidx, aidx = {c: i for i, c in enumerate(CLASSES)}, {a: i for i, a in enumerate(PREDS)}
missing_unseen = UNSEEN - set(CLASSES)
if missing_unseen:
    print("note: unseen-split names not found:", sorted(missing_unseen))
UNSEEN = UNSEEN & set(CLASSES)
SEEN = [c for c in CLASSES if c not in UNSEEN]
pretty = lambda c: c.replace("+", " ")
n_cls, n_att = len(CLASSES), len(PREDS)

def center_square(img):
    w, h = img.size; s = min(w, h)
    return img.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s))

rng = random.Random(SEED)
chosen = [(c, n) for c in SEEN for n in rng.sample(sorted(META["images"][c]),
                                                   min(N_PER_CLASS, len(META["images"][c])))]
_local = threading.local()
def fetch(item):
    c, n = item
    out = os.path.join(IMG_DIR, os.path.basename(n))
    if not os.path.exists(out):
        if not hasattr(_local, "z"):
            _local.z = open_zip(AWA_DATA)[0]
        img = Image.open(io.BytesIO(_local.z.read(n))).convert("RGB")
        center_square(img).resize((512, 512), Image.LANCZOS).save(out, quality=92)
    return out

todo = [it for it in chosen if not os.path.exists(os.path.join(IMG_DIR, os.path.basename(it[1])))]
if todo:
    workers = 8 if open_zip(AWA_DATA)[1] == "remote" else 1
    with cf.ThreadPoolExecutor(workers) as ex:
        list(ex.map(fetch, todo))
MEM_PATHS = [os.path.join(IMG_DIR, os.path.basename(n)) for _, n in chosen]
MEM_CLASS = np.array([cidx[c] for c, _ in chosen])
n_img = len(MEM_PATHS)
print(f"{n_cls} animals ({len(SEEN)} with images), {n_att} attributes, memory: {n_img} images")

### 4. CLIP (small one for retrieval, large one only for judging)

In [ ]:
from transformers import CLIPModel, CLIPProcessor

def load_clip(name):
    m = CLIPModel.from_pretrained(name, torch_dtype=torch.float16).to(DEVICE).eval()
    return m, CLIPProcessor.from_pretrained(name)

def _feat(out, kind):
    if torch.is_tensor(out):
        return out
    for k in (f"{kind}_embeds", "pooler_output"):
        if getattr(out, k, None) is not None:
            return getattr(out, k)
    return out[0]

@torch.no_grad()
def clip_images(model, proc, imgs, bs=64):
    feats = []
    for i in range(0, len(imgs), bs):
        batch = [Image.open(x).convert("RGB") if isinstance(x, str) else x for x in imgs[i:i + bs]]
        pv = proc(images=batch, return_tensors="pt")["pixel_values"].to(DEVICE, torch.float16)
        feats.append(_feat(model.get_image_features(pixel_values=pv), "image").float())
    return F.normalize(torch.cat(feats), dim=1)

@torch.no_grad()
def clip_texts(model, proc, texts):
    tok = proc.tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
    return F.normalize(_feat(model.get_text_features(**tok), "text").float(), dim=1)

clip_r, proc_r = load_clip(CLIP_RETRIEVE)
clip_j, proc_j = load_clip(CLIP_JUDGE)
emb_path = os.path.join(OUT, f"mem_clipB32_{n_img}.pt")
if os.path.exists(emb_path):
    MEM_EMB = torch.load(emb_path).to(DEVICE)
else:
    MEM_EMB = clip_images(clip_r, proc_r, MEM_PATHS); torch.save(MEM_EMB.cpu(), emb_path)
print("memory embeddings:", tuple(MEM_EMB.shape))

### 5. Attribute vocabulary, categories and the query parser

In [ ]:
CATEGORY = {**{c: "color" for c in COLORS},
            **{a: "pattern" for a in ("stripes", "spots", "patches", "furry", "hairless")},
            **{a: "size" for a in ("big", "small", "bulbous", "lean")},
            **{a: "part" for a in ("flippers", "hooves", "longneck", "longleg", "tail", "horns",
                                   "claws", "tusks", "flys", "bipedal", "quadrapedal", "paws", "hands")},
            **{a: "scene" for a in ("arctic", "desert", "jungle", "forest", "ocean", "mountains",
                                    "plains", "water", "tree", "cave", "coastal", "fields", "bush")}}
CATEGORY = {a: c for a, c in CATEGORY.items() if a in aidx}
VISUAL = np.array([aidx[a] for a in CATEGORY])

WORD2ATTR = {**{c: c for c in COLORS}, "grey": "gray", "golden": "yellow",
             "striped": "stripes", "stripes": "stripes", "stripy": "stripes",
             "spotted": "spots", "spots": "spots", "spotty": "spots",
             "patchy": "patches", "patches": "patches", "furry": "furry", "fluffy": "furry",
             "hairless": "hairless", "bald": "hairless",
             "horn": "horns", "horns": "horns", "horned": "horns", "tusk": "tusks", "tusks": "tusks",
             "flipper": "flippers", "flippers": "flippers", "hoof": "hooves", "hooves": "hooves",
             "hoofed": "hooves", "claws": "claws", "clawed": "claws",
             "long neck": "longneck", "long-necked": "longneck", "long legs": "longleg",
             "long-legged": "longleg", "tail": "tail", "wings": "flys", "winged": "flys", "flying": "flys",
             "big": "big", "huge": "big", "giant": "big", "small": "small", "tiny": "small",
             "fat": "bulbous", "chubby": "bulbous", "thin": "lean", "slim": "lean",
             "arctic": "arctic", "snow": "arctic", "snowy": "arctic", "desert": "desert",
             "jungle": "jungle", "forest": "forest", "woods": "forest", "ocean": "ocean", "sea": "ocean",
             "underwater": "water", "mountain": "mountains", "mountains": "mountains",
             "grassland": "plains", "plains": "plains", "cave": "cave", "beach": "coastal"}
WORD2ATTR = {w: a for w, a in WORD2ATTR.items() if a in aidx}
ALIASES = {"cat": "persian+cat", "bear": "grizzly+bear", "whale": "blue+whale", "panda": "giant+panda",
           "monkey": "spider+monkey", "dog": "german+shepherd", "mouse": "mouse", "deer": "deer"}
ANIMAL_NAMES = {}
for c in CLASSES:
    for v in {pretty(c), pretty(c) + "s", pretty(c) + "es"}:
        ANIMAL_NAMES[v] = c
for k, c in ALIASES.items():
    if c in cidx:
        ANIMAL_NAMES.setdefault(k, c); ANIMAL_NAMES.setdefault(k + "s", c)
NUMBERS = {w: i for i, w in enumerate("zero one two three four five six seven eight nine ten".split())}

def parse_query(q):
    """-> animal (or None), known attributes, unknown edits the graph cannot represent."""
    text = " " + re.sub(r"[^a-z0-9\- ]", " ", q.lower()) + " "
    animal = None
    for name in sorted(ANIMAL_NAMES, key=len, reverse=True):
        m = re.search(rf"(?<![a-z]){re.escape(name)}(?![a-z])", text)
        if m:
            animal = ANIMAL_NAMES[name]; text = text[:m.start()] + " " + text[m.end():]
            break
    attrs, unknown, part_colors = [], [], []
    colour_words = [w for w, a in WORD2ATTR.items() if CATEGORY.get(a) == "color"]
    part_words = [w for w, a in WORD2ATTR.items() if CATEGORY.get(a) == "part"]
    pat = rf"(?<![a-z])({'|'.join(map(re.escape, colour_words))})\s+({'|'.join(map(re.escape, part_words))})(?![a-z])"
    for m in re.finditer(pat, text):                     # "blue horns": colour of a part, not of the animal
        part = WORD2ATTR[m.group(2)]
        attrs.append(part); part_colors.append(f"{m.group(1)} {m.group(2)}")
    text = re.sub(pat, " ", text)
    for m in re.finditer(r"(\w+)[- ]legged|(\w+) legs", text):
        word = m.group(1) or m.group(2)
        n = int(word) if word.isdigit() else NUMBERS.get(word)
        if n == 2 and "bipedal" in aidx: attrs.append("bipedal")
        elif n == 4 and "quadrapedal" in aidx: attrs.append("quadrapedal")
        elif n is not None: unknown.append(f"{n} legs")
    text = re.sub(r"(\w+)[- ]legged|(\w+) legs", " ", text)
    for w in sorted(WORD2ATTR, key=len, reverse=True):
        if re.search(rf"(?<![a-z]){re.escape(w)}(?![a-z])", text):
            attrs.append(WORD2ATTR[w]); text = re.sub(rf"(?<![a-z]){re.escape(w)}(?![a-z])", " ", text)
    return dict(animal=animal, attrs=list(dict.fromkeys(attrs)), unknown=unknown, part_colors=part_colors)

for q in ("a blue tiger", "a striped horse in the snow", "an elephant with horns",
          "a five-legged dog", "an animal with stripes and hooves", "a red blue whale",
          "an elephant with blue horns"):
    print(f"{q!r:40} -> {parse_query(q)}")

### 6. GNN: training-free message passing on the image–animal–attribute graph

In [ ]:
# Layer 1 (images -> animals): each animal's visual-attribute profile is its AwA2 prior (70%)
#   mixed with the CLIP evidence of its images (30%). Layer 2 (animals -> images): each image's
#   profile mixes its animal's profile with its own evidence (weight MP_BETA by attribute type).
#   Unseen animals keep their prior only. No weights are trained.
ATTR_PHRASE = {**{c: c for c in COLORS}, "stripes": "striped", "spots": "spotted", "patches": "patchy",
               "furry": "furry", "hairless": "hairless", "big": "big", "small": "small", "bulbous": "fat",
               "lean": "lean", "flippers": "with flippers", "hooves": "with hooves",
               "longneck": "with a long neck", "longleg": "with long legs", "tail": "with a tail",
               "horns": "with horns", "claws": "with claws", "tusks": "with tusks", "flys": "with wings",
               "bipedal": "standing on two legs", "quadrapedal": "on four legs", "paws": "with paws",
               "hands": "with hands", "arctic": "in the snow", "desert": "in the desert",
               "jungle": "in the jungle", "forest": "in a forest", "ocean": "in the ocean",
               "mountains": "in the mountains", "plains": "on grassland", "water": "in water",
               "tree": "in a tree", "cave": "in a cave", "coastal": "on a beach", "fields": "in a field",
               "bush": "in the bush"}
vis_names = [PREDS[i] for i in VISUAL]
ADJ_CATS = {"color", "pattern", "size"}            # "a photo of a blue animal", not "an animal blue"
T_ATTR = clip_texts(clip_r, proc_r, [f"a photo of a {ATTR_PHRASE.get(a, a)} animal" if CATEGORY[a] in ADJ_CATS
                                     else f"a photo of an animal {ATTR_PHRASE.get(a, a)}" for a in vis_names])
raw = (MEM_EMB @ T_ATTR.T).cpu().numpy()                              # (n_img, n_visual)
z = (raw - raw.mean(0)) / (raw.std(0) + 1e-6)
# Only strong evidence counts (about the top 7% of photos per attribute), so the planner
# cannot exploit noise by claiming that an ordinary photo already shows the attribute.
EVID = np.zeros((n_img, n_att)); EVID[:, VISUAL] = 1 / (1 + np.exp(-3 * (z - 1.5)))

t0 = time.perf_counter()
A_ic = sp.csr_matrix((np.ones(n_img), (np.arange(n_img), MEM_CLASS)), shape=(n_img, n_cls))
cnt = np.asarray(A_ic.sum(0)).ravel()
img_mean = (A_ic.T @ EVID) / np.maximum(cnt, 1)[:, None]               # message: images -> animals
H_CLS = ATTR.copy()
has = cnt > 0
H_CLS[np.ix_(has, VISUAL)] = 0.7 * ATTR[np.ix_(has, VISUAL)] + 0.3 * img_mean[np.ix_(has, VISUAL)]
BETA = np.zeros(n_att)
for a, c in CATEGORY.items():
    BETA[aidx[a]] = MP_BETA[c]
H_IMG = (1 - BETA) * (A_ic @ H_CLS) + BETA * EVID                     # message: animals -> images
GNN_PRECOMPUTE_S = time.perf_counter() - t0
print(f"message passing done in {GNN_PRECOMPUTE_S * 1e3:.1f} ms; image profiles {H_IMG.shape}")

### 7. Retrieval + planning for both RAG pipelines (uses the calibrated table if it exists)

In [ ]:
CAL_PATH = os.path.join(OUT, "calibration.json")
if os.path.exists(CAL_PATH) and not RECALIBRATE:
    STRENGTH.update(json.load(open(CAL_PATH))["strength"])
    print("using CALIBRATED noise levels:", STRENGTH)
else:
    print("using INITIAL (uncalibrated) noise levels:", STRENGTH)
@torch.no_grad()
def text_emb(q):
    return clip_texts(clip_r, proc_r, [q])[0]

def retrieve_normal(query):
    t0 = time.perf_counter()
    idx = int((MEM_EMB @ text_emb(query)).argmax())
    return idx, (time.perf_counter() - t0) * 1e3

def missing_edits(parse, idx):
    miss = []
    if parse["animal"] and cidx[parse["animal"]] != MEM_CLASS[idx]:
        miss.append(("species", parse["animal"]))
    coloured = {WORD2ATTR[pc.split()[-1]] for pc in parse.get("part_colors", [])}
    for a in parse["attrs"]:
        if a not in coloured and H_IMG[idx, aidx[a]] < 0.5:
            miss.append((CATEGORY.get(a, "part"), a))
    miss += [("part", pc) for pc in parse.get("part_colors", [])]   # one edit: add/colour the part
    miss += [("unknown", u) for u in parse["unknown"]]
    return miss

def edit_strength(miss):
    return max([STRENGTH[c] for c, _ in miss], default=STRENGTH["none"])

def edit_costs(parse):
    """Planned noise strength and number of edits for EVERY memory image (vectorised missing_edits)."""
    cost = np.full(n_img, STRENGTH["none"]); n_edits = np.zeros(n_img, dtype=int)
    if parse["animal"]:
        other = MEM_CLASS != cidx[parse["animal"]]
        cost = np.where(other, np.maximum(cost, STRENGTH["species"]), cost); n_edits += other
    coloured = {WORD2ATTR[pc.split()[-1]] for pc in parse.get("part_colors", [])}
    for a in parse["attrs"]:
        if a in coloured:
            continue
        miss = H_IMG[:, aidx[a]] < 0.5
        cost = np.where(miss, np.maximum(cost, STRENGTH[CATEGORY.get(a, "part")]), cost); n_edits += miss
    if parse.get("part_colors"):
        cost = np.maximum(cost, STRENGTH["part"]); n_edits += len(parse["part_colors"])
    if parse["unknown"]:
        cost = np.maximum(cost, STRENGTH["unknown"]); n_edits += len(parse["unknown"])
    return cost, n_edits

def plan_gnn(query, fixed_idx=None):
    """Retrieve the image needing the smallest edit, then set noise strength from what is missing."""
    t0 = time.perf_counter()
    parse = parse_query(query)
    s_clip = (MEM_EMB @ text_emb(query)).cpu().numpy()
    if fixed_idx is None:
        req = [aidx[a] for a in parse["attrs"]]
        t = ATTR[cidx[parse["animal"]]].copy() if parse["animal"] else np.zeros(n_att)
        t[req] = 1.0
        w = np.full(n_att, 0.3); w[req] = 1.0
        Hw, tw = H_IMG * w, t * w
        s_attr = Hw @ tw / (np.linalg.norm(Hw, axis=1) * np.linalg.norm(tw) + 1e-9)
        score = s_attr + ALPHA_CLIP * s_clip
        cost, n_edits = edit_costs(parse)
        # lowest noise first; on ties, fewest changes (a species change counts), then best match
        idx = int(np.lexsort((-score, n_edits, cost))[0])
    else:
        idx = int(fixed_idx)
    miss = missing_edits(parse, idx)
    s = edit_strength(miss)
    color = next((a for c, a in miss if c == "color"), None)
    return dict(parse=parse, idx=idx, retrieved_animal=CLASSES[MEM_CLASS[idx]], missing=miss,
                strength=s, steps=int(N_NOMINAL * s), recolor=color,
                retrieval_ms=(time.perf_counter() - t0) * 1e3)

for q in ("a blue tiger", "a striped horse", "a tiger in the arctic", "an animal with stripes and hooves",
          "a five-legged dog", "a red leopard", "an elephant with blue horns"):
    p = plan_gnn(q)
    print(f"{q!r:38} -> start from {pretty(p['retrieved_animal']):>15}, missing {p['missing']}, "
          f"strength {p['strength']} = {p['steps']}/{N_NOMINAL} steps")

### 8. Recolour, Stable Diffusion (DDIM and DPM-Solver++), SD-Turbo, and the three pipelines

In [ ]:
from matplotlib.colors import rgb_to_hsv, hsv_to_rgb
HUE = {"red": 0.0, "orange": 0.08, "yellow": 0.15, "blue": 0.6}
def recolor(img, color):
    hsv = rgb_to_hsv(np.asarray(img, dtype=np.float32) / 255.0)
    if color in HUE:
        hsv[..., 0] = HUE[color]; hsv[..., 1] = np.maximum(hsv[..., 1], 0.55)
    elif color == "brown":
        hsv[..., 0] = 0.07; hsv[..., 1] = np.maximum(hsv[..., 1], 0.5); hsv[..., 2] *= 0.7
    elif color == "gray":
        hsv[..., 1] *= 0.1
    elif color == "black":
        hsv[..., 1] *= 0.3; hsv[..., 2] *= 0.35
    elif color == "white":
        hsv[..., 1] *= 0.1; hsv[..., 2] = 0.6 + 0.4 * hsv[..., 2]
    return Image.fromarray((hsv_to_rgb(hsv) * 255).clip(0, 255).astype(np.uint8))

from diffusers import (StableDiffusionPipeline, StableDiffusionImg2ImgPipeline, DDIMScheduler,
                       DPMSolverMultistepScheduler, AutoPipelineForText2Image)
kw = dict(torch_dtype=torch.float16, safety_checker=None, requires_safety_checker=False)
try:
    base = StableDiffusionPipeline.from_pretrained(SD_ID, variant="fp16", **kw)
except Exception:
    base = StableDiffusionPipeline.from_pretrained(SD_ID, **kw)
base = base.to(DEVICE)
comps = {k: v for k, v in base.components.items() if k != "scheduler"}
def sched(kind):
    if kind == "ddim":
        return DDIMScheduler.from_config(base.scheduler.config)
    return DPMSolverMultistepScheduler.from_config(base.scheduler.config, algorithm_type="dpmsolver++")
PIPES = {}
for kind in ("ddim", "dpm"):                         # same weights, separate schedulers
    PIPES[("t2i", kind)] = StableDiffusionPipeline(**comps, scheduler=sched(kind))
    PIPES[("i2i", kind)] = StableDiffusionImg2ImgPipeline(**comps, scheduler=sched(kind))
for p_ in PIPES.values():
    p_.set_progress_bar_config(disable=True)

turbo = None
if USE_TURBO:
    try:
        turbo = AutoPipelineForText2Image.from_pretrained(TURBO_ID, torch_dtype=torch.float16, variant="fp16")
    except Exception:
        turbo = AutoPipelineForText2Image.from_pretrained(TURBO_ID, torch_dtype=torch.float16)
    turbo = turbo.to(DEVICE); turbo.set_progress_bar_config(disable=True)

def sync():
    if DEVICE == "cuda": torch.cuda.synchronize()

def _counter():
    count = [0]
    def cb(pipe, i, t, kwargs):
        count[0] += 1
        return kwargs
    return count, cb

def generate(prompt, seed, steps=N_NOMINAL, init=None, strength=None, sampler="ddim"):
    """Returns image, denoising steps actually run, seconds."""
    count, cb = _counter()
    g = torch.Generator(DEVICE).manual_seed(seed)
    sync(); t0 = time.perf_counter()
    if init is None:
        img = PIPES[("t2i", sampler)](prompt, num_inference_steps=steps, guidance_scale=GUIDANCE,
                                      generator=g, callback_on_step_end=cb).images[0]
    else:
        img = PIPES[("i2i", sampler)](prompt, image=init, strength=strength, num_inference_steps=N_NOMINAL,
                                      guidance_scale=GUIDANCE, generator=g, callback_on_step_end=cb).images[0]
    sync()
    return img, count[0], time.perf_counter() - t0

def generate_turbo(prompt, seed, steps):
    count, cb = _counter()
    g = torch.Generator(DEVICE).manual_seed(seed)
    sync(); t0 = time.perf_counter()
    img = turbo(prompt, num_inference_steps=steps, guidance_scale=0.0, generator=g,
                callback_on_step_end=cb).images[0]
    sync()
    return img, count[0], time.perf_counter() - t0

def mem_image(idx):
    return Image.open(MEM_PATHS[idx]).convert("RGB")

def _res(img, steps, secs, idx=None, init=None, ms=0.0, plan=None):
    return dict(img=img, init=init, idx=idx, steps=steps, retrieval_ms=ms, gen_s=secs,
                total_s=secs + ms / 1e3, plan=plan)

def run_plain(prompt, seed, steps=N_NOMINAL, sampler="ddim"):
    img, n, secs = generate(prompt, seed, steps, sampler=sampler)
    return _res(img, n, secs)

def run_turbo(prompt, seed, steps):
    img, n, secs = generate_turbo(prompt, seed, steps)
    return _res(img, n, secs)

def run_normal_rag(prompt, seed, sampler="ddim"):
    idx, ms = retrieve_normal(prompt)
    img, n, secs = generate(prompt, seed, init=mem_image(idx), strength=FIXED_STRENGTH, sampler=sampler)
    return _res(img, n, secs, idx, mem_image(idx), ms)

def run_gnn_rag(prompt, seed, plan=None, sampler="ddim"):
    plan = plan or plan_gnn(prompt)
    init = mem_image(plan["idx"])
    if plan["recolor"]:
        init = recolor(init, plan["recolor"])
    img, n, secs = generate(prompt, seed, init=init, strength=plan["strength"], sampler=sampler)
    return _res(img, n, secs, plan["idx"], init, plan["retrieval_ms"], plan)

def run_fixed_from(prompt, seed, plan):
    """Ablation: GNN-retrieved start image, fixed strength of normal RAG, no edit."""
    img, n, secs = generate(prompt, seed, init=mem_image(plan["idx"]), strength=FIXED_STRENGTH)
    return _res(img, n, secs, plan["idx"], mem_image(plan["idx"]), plan["retrieval_ms"])

def run_clip_planned(prompt, seed):
    """Ablation: CLIP-retrieved start image, then the graph's plan (strength + recolour)."""
    idx, ms = retrieve_normal(prompt)
    plan = plan_gnn(prompt, fixed_idx=idx)
    plan["retrieval_ms"] += ms
    return run_gnn_rag(prompt, seed, plan=plan)

generate("a photo of a cat", 0, 5); generate("a photo of a cat", 0, 5, sampler="dpm")   # warm-up
if turbo is not None:
    generate_turbo("a photo of a cat", 0, 1)

### 9. Judging and statistics helpers

In [ ]:
@torch.no_grad()
def judge(img, prompt, base_text):
    e = clip_images(clip_j, proc_j, [img])[0]
    t = clip_texts(clip_j, proc_j, [prompt, base_text])
    return float(100 * e @ t[0]), float(100 * (e @ t[0] - e @ t[1])), e

T_ANIMAL = clip_texts(clip_j, proc_j, [f"a photo of {'an' if pretty(c)[0] in 'aeiou' else 'a'} {pretty(c)}"
                                       for c in CLASSES])

@torch.no_grad()
def judge_rows(rows, prompts_by_id):
    """Adds clip_score, attr_gain, animal_ok to each row (large CLIP, batched)."""
    E = clip_images(clip_j, proc_j, [r["path"] for r in rows])
    texts = sorted({t for r in rows for t in (prompts_by_id[r["prompt_id"]]["text"],
                                              prompts_by_id[r["prompt_id"]]["base"])})
    T = clip_texts(clip_j, proc_j, texts); ti = {t: i for i, t in enumerate(texts)}
    for r, e in zip(rows, E):
        p = prompts_by_id[r["prompt_id"]]
        s_p, s_b = float(e @ T[ti[p["text"]]]), float(e @ T[ti[p["base"]]])
        r["clip_score"], r["attr_gain"] = 100 * s_p, 100 * (s_p - s_b)
        r["animal_ok"] = bool(CLASSES[int((e @ T_ANIMAL.T).argmax())] in p["valid"])

def cluster_ci(values_by_prompt, level=0.95, n=5000):
    """values_by_prompt: one value per prompt (already averaged over its seeds). Bootstrap over prompts."""
    v = np.asarray(values_by_prompt, dtype=float)
    rs = np.random.RandomState(SEED)
    boots = v[rs.randint(0, len(v), (n, len(v)))].mean(1)
    a = (1 - level) / 2
    return float(v.mean()), np.percentile(boots, [100 * a, 100 * (1 - a)]).round(3).tolist()

## Part A0 — Calibrate the noise-level table (separate prompts, never the benchmark's)
For every kind of change, GNN-RAG is run at noise strengths 0.4–0.9 on calibration prompts and
compared with plain diffusion (25 steps). The calibrated strength is the **lowest** one whose
mean CLIP score is within 0.5 points of plain diffusion — the rule is fixed before looking.

### 10. Prompt sets: benchmark and (disjoint) calibration

In [ ]:
ADJ   = {"stripes": "striped", "spots": "spotted", "patches": "patchy"}
WITH  = {"horns": "horns", "tusks": "tusks", "flippers": "flippers", "longneck": "a long neck",
         "hooves": "hooves", "stripes": "stripes", "spots": "spots", "patches": "patches"}
SCENE = {"arctic": "in the arctic", "desert": "in the desert", "jungle": "in the jungle",
         "ocean": "in the ocean", "mountains": "in the mountains", "forest": "in a forest"}
art = lambda w: "an" if w[0] in "aeiou" else "a"
EDIT_POOLS = {
    "color":   (COLORS, lambda c, a: f"a photo of {art(a)} {a} {pretty(c)}"),
    "pattern": (list(ADJ), lambda c, a: f"a photo of {art(ADJ[a])} {ADJ[a]} {pretty(c)}"),
    "part":    (["horns", "tusks", "flippers", "longneck", "hooves"],
                lambda c, a: f"a photo of {art(pretty(c))} {pretty(c)} with {WITH[a]}"),
    "scene":   (list(SCENE), lambda c, a: f"a photo of {art(pretty(c))} {pretty(c)} {SCENE[a]}"),
}

def lacking(pool, exclude):
    return [(c, a) for c in CLASSES for a in pool
            if a in aidx and BINARY[cidx[c], aidx[a]] == 0 and (c, a) not in exclude]

def base_text(c):
    return f"a photo of {art(pretty(c))} {pretty(c)}"

def build_benchmark():
    rs = random.Random(SEED)
    out = []
    for cat, (pool, fmt) in EDIT_POOLS.items():
        for c, a in rs.sample(lacking(pool, set()), N_PER_CATEGORY):
            out.append(dict(category=cat, animal=c, attrs=[a], text=fmt(c, a), base=base_text(c), valid=[c]))
    pool = [a for a in ("stripes", "spots", "patches", "horns", "tusks", "flippers", "longneck", "hooves") if a in aidx]
    pairs = [(a, b) for i, a in enumerate(pool) for b in pool[i + 1:]
             if any(BINARY[k, aidx[a]] and BINARY[k, aidx[b]] for k in range(n_cls))]
    for a, b in rs.sample(pairs, min(10, len(pairs))):
        valid = [c for c in CLASSES if BINARY[cidx[c], aidx[a]] and BINARY[cidx[c], aidx[b]]]
        out.append(dict(category="attr-only", animal=None, attrs=[a, b],
                        text=f"a photo of an animal with {WITH[a]} and {WITH[b]}",
                        base="a photo of an animal", valid=valid))
    for i, p in enumerate(out):
        p["id"] = i
    return out

def build_calibration(bench):
    rs = random.Random(SEED + 1000)
    used = {(p["animal"], a) for p in bench for a in p["attrs"]}
    out = []
    for cat, (pool, fmt) in EDIT_POOLS.items():          # seen animals: start from the SAME animal
        for c, a in rs.sample([x for x in lacking(pool, used) if x[0] in SEEN], N_CAL):
            out.append(dict(intended=cat, animal=c, attrs=[a], text=fmt(c, a), base=base_text(c), valid=[c]))
    for c in rs.sample(SEEN, N_CAL):                                      # nothing to change
        out.append(dict(intended="none", animal=c, attrs=[], text=base_text(c), base=base_text(c), valid=[c]))
    for c in rs.sample(sorted(UNSEEN), min(N_CAL, len(UNSEEN))):          # animal not in memory
        out.append(dict(intended="species", animal=c, attrs=[], text=base_text(c), base=base_text(c), valid=[c]))
    for i, p in enumerate(out):
        p["id"] = i
    return out

PROMPTS = build_benchmark()
CAL_PROMPTS = build_calibration(PROMPTS)
assert not ({p["text"] for p in PROMPTS} & {p["text"] for p in CAL_PROMPTS}), "calibration overlaps benchmark"
bad = [p["text"] for p in PROMPTS if parse_query(p["text"])["animal"] != p["animal"]
       or set(parse_query(p["text"])["attrs"]) != set(p["attrs"])]
print(f"benchmark: {len(PROMPTS)} prompts × {N_SEEDS} seeds | calibration: {len(CAL_PROMPTS)} prompts | "
      f"parser errors on benchmark: {len(bad)}", bad[:5])

### 11. Run calibration (resumable) and set the noise-level table

In [ ]:
def dominant(missing, table):
    return max(missing, key=lambda m: table[m[0]])[0] if missing else "none"

def calibration_plan(p):
    """Isolate one kind of change: start from the requested animal's own best-matching photo
    (so the only change is the intended one); unseen animals use the normal plan (species change)."""
    if p["intended"] == "species":
        return plan_gnn(p["text"])
    own = np.where(MEM_CLASS == cidx[p["animal"]])[0]
    idx = int(own[np.argmax((MEM_EMB[torch.as_tensor(own, device=DEVICE)] @ text_emb(p["text"])).cpu().numpy())])
    return plan_gnn(p["text"], fixed_idx=idx)

if os.path.exists(CAL_PATH) and not RECALIBRATE:
    CAL = json.load(open(CAL_PATH))
    print("calibration already done:", CAL["strength"])
else:
    STRENGTH.clear(); STRENGTH.update(INITIAL_STRENGTH)           # plan with the initial table
    cal_rows_path = os.path.join(OUT, "calibration_rows.jsonl")
    cal_dir = os.path.join(OUT, "calibration_images"); os.makedirs(cal_dir, exist_ok=True)
    done = {}
    if os.path.exists(cal_rows_path):
        for line in open(cal_rows_path):
            r = json.loads(line); done[r["key"]] = r
    for p in CAL_PROMPTS:
        seed = SEED + 5000 + p["id"]
        plan = calibration_plan(p)
        dom = dominant(plan["missing"], INITIAL_STRENGTH)
        jobs = [("plain_25", None)] + [(f"s{s}", s) for s in CAL_STRENGTHS]
        for name, s in jobs:
            key = f"{p['id']}|{name}"
            if key in done:
                continue
            res = run_plain(p["text"], seed) if s is None else run_gnn_rag(p["text"], seed, plan={**plan, "strength": s})
            path = os.path.join(cal_dir, f"{p['id']}_{name}.jpg"); res["img"].save(path, quality=92)
            row = dict(key=key, prompt_id=p["id"], name=name, strength=s, category=dom,
                       steps=res["steps"], path=path)
            done[key] = row
            with open(cal_rows_path, "a") as f:
                f.write(json.dumps(row) + "\n")
        print(f"calibration {p['id'] + 1}/{len(CAL_PROMPTS)}: {p['text']!r} -> {dom}")
    cal_rows = list(done.values())
    judge_rows(cal_rows, {p["id"]: p for p in CAL_PROMPTS})
    ref = {r["prompt_id"]: r["clip_score"] for r in cal_rows if r["name"] == "plain_25"}
    curves, new = {}, {}
    for cat in sorted({r["category"] for r in cal_rows}):
        ids = {r["prompt_id"] for r in cal_rows if r["category"] == cat}
        curves[cat] = {s: float(np.mean([r["clip_score"] - ref[r["prompt_id"]] for r in cal_rows
                                         if r["category"] == cat and r["strength"] == s])) for s in CAL_STRENGTHS}
        ok_s = [s for s in CAL_STRENGTHS if curves[cat][s] >= -CAL_TOL]
        if len(ids) < 3:
            print(f"  {cat}: only {len(ids)} calibration prompts -> keeping initial value")
            continue
        new[cat] = min(ok_s) if ok_s else max(curves[cat], key=curves[cat].get)
        print(f"  {cat:>8} ({len(ids)} prompts): Δ CLIP vs plain by strength "
              f"{ {s: round(v, 2) for s, v in curves[cat].items()} } -> {new[cat]}")
    table = dict(INITIAL_STRENGTH); table.update(new)
    table["size"] = table.get("pattern", table["size"]) if "size" not in new else table["size"]
    table["unknown"] = max(0.9, max(v for k, v in table.items() if k != "unknown"))
    CAL = dict(strength=table, curves=curves, tol=CAL_TOL, strengths=CAL_STRENGTHS)
    json.dump(CAL, open(CAL_PATH, "w"), indent=2)
    import matplotlib.pyplot as plt
    plt.figure(figsize=(6, 4))
    for cat, cv in curves.items():
        plt.plot(list(cv), list(cv.values()), marker="o", label=cat)
    plt.axhline(-CAL_TOL, ls="--", c="gray"); plt.xlabel("noise strength"); plt.ylabel("Δ CLIP score vs plain (25 steps)")
    plt.legend(); plt.title("Calibration"); plt.tight_layout()
    plt.savefig(os.path.join(OUT, "fig_calibration.png"), dpi=150); plt.show()
STRENGTH.clear(); STRENGTH.update(CAL["strength"])
print("noise levels used from now on:", STRENGTH)

## Part A — Benchmark
Conditions per (prompt, seed): `plain_25` · `plain_matched` (DDIM, same steps as GNN-RAG) ·
`plain_dpm_matched` (DPM-Solver++, same steps) · `normal_rag` · `gnn_rag` · `gnn_rag_dpm` ·
`abl_gnn_fixed` (GNN retrieval, no planning) · `abl_clip_planned` (CLIP retrieval + graph plan) ·
`turbo_1`, `turbo_4` (SD-Turbo). Statistics bootstrap over **prompts** (seeds averaged inside).

### 12. Generate the benchmark (resumable; tied to this exact noise-level table)

In [ ]:
import hashlib
TABLE_ID = hashlib.md5(json.dumps(STRENGTH, sort_keys=True).encode()).hexdigest()[:8]
GEN_DIR3 = os.path.join(OUT, f"generated_{TABLE_ID}"); os.makedirs(GEN_DIR3, exist_ok=True)
rows_path = os.path.join(OUT, f"rows_{TABLE_ID}.jsonl")
CONDS = ["plain_25", "plain_matched", "plain_dpm_matched", "normal_rag", "gnn_rag", "gnn_rag_dpm",
         "abl_gnn_fixed", "abl_clip_planned"] + (["turbo_1", "turbo_4"] if turbo is not None else [])
DONE = {}
if os.path.exists(rows_path):
    for line in open(rows_path):
        r = json.loads(line); DONE[r["key"]] = r
total = len(PROMPTS) * N_SEEDS * len(CONDS)
print(f"table {TABLE_ID}: {len(DONE)}/{total} images done")

t_start = time.perf_counter()
for p in PROMPTS:
    plan = plan_gnn(p["text"])
    for k in range(N_SEEDS):
        seed = SEED + 10 * p["id"] + k
        jobs = {
            "plain_25":          lambda: run_plain(p["text"], seed),
            "plain_matched":     lambda: run_plain(p["text"], seed, steps=plan["steps"]),
            "plain_dpm_matched": lambda: run_plain(p["text"], seed, steps=plan["steps"], sampler="dpm"),
            "normal_rag":        lambda: run_normal_rag(p["text"], seed),
            "gnn_rag":           lambda: run_gnn_rag(p["text"], seed, plan=plan),
            "gnn_rag_dpm":       lambda: run_gnn_rag(p["text"], seed, plan=plan, sampler="dpm"),
            "abl_gnn_fixed":     lambda: run_fixed_from(p["text"], seed, plan),
            "abl_clip_planned":  lambda: run_clip_planned(p["text"], seed),
            "turbo_1":           lambda: run_turbo(p["text"], seed, 1),
            "turbo_4":           lambda: run_turbo(p["text"], seed, 4),
        }
        for cond in CONDS:
            key = f"{p['id']}|{k}|{cond}"
            if key in DONE:
                continue
            res = jobs[cond]()
            path = os.path.join(GEN_DIR3, f"{p['id']}_{k}_{cond}.jpg"); res["img"].save(path, quality=92)
            row = dict(key=key, prompt_id=p["id"], seed_k=k, cond=cond, category=p["category"],
                       steps=res["steps"], retrieval_ms=res["retrieval_ms"], gen_s=res["gen_s"],
                       total_s=res["total_s"], path=path,
                       retrieved=None if res["idx"] is None else CLASSES[MEM_CLASS[res["idx"]]],
                       strength=res["plan"]["strength"] if res["plan"] else None,
                       missing=res["plan"]["missing"] if res["plan"] else None)
            DONE[key] = row
            with open(rows_path, "a") as f:
                f.write(json.dumps(row) + "\n")
    print(f"prompt {p['id'] + 1}/{len(PROMPTS)} ({len(DONE)}/{total}, {(time.perf_counter() - t_start) / 60:.1f} min)")

### 13. Judge every image with the large CLIP

In [ ]:
rows = [DONE[f"{p['id']}|{k}|{c}"] for p in PROMPTS for k in range(N_SEEDS) for c in CONDS]
PBY = {p["id"]: p for p in PROMPTS}
judge_rows(rows, PBY)
BY = {(r["prompt_id"], r["seed_k"], r["cond"]): r for r in rows}
print("judged", len(rows), "images")

### 14. Results: tables, primary comparisons, figure, CSV for the paper

In [ ]:
import csv
import matplotlib.pyplot as plt

def per_prompt(cond, metric, cat=None):
    return [np.mean([float(BY[(p["id"], k, cond)][metric]) for k in range(N_SEEDS)])
            for p in PROMPTS if cat is None or p["category"] == cat]

def per_prompt_diff(a, b, metric, cat=None):
    return [np.mean([float(BY[(p["id"], k, a)][metric]) - float(BY[(p["id"], k, b)][metric])
                     for k in range(N_SEEDS)]) for p in PROMPTS if cat is None or p["category"] == cat]

SUMMARY = []
for cat in (None, "color", "pattern", "part", "scene", "attr-only"):
    print(f"\n=== {cat or 'all prompts'} ===")
    print(f"{'condition':>18} {'CLIP [95% CI]':>22} {'animal':>7} {'attr+':>6} {'steps':>6} {'total s':>8}")
    for c in CONDS:
        m, ci = cluster_ci(per_prompt(c, "clip_score", cat))
        an = np.mean(per_prompt(c, "animal_ok", cat)); ag = np.mean(per_prompt(c, "attr_gain", cat))
        st = np.mean(per_prompt(c, "steps", cat)); ts = np.mean(per_prompt(c, "total_s", cat))
        print(f"{c:>18} {m:7.2f} [{ci[0]:6.2f},{ci[1]:6.2f}] {an:7.2f} {ag:6.2f} {st:6.1f} {ts:8.2f}")
        SUMMARY.append(dict(category=cat or "all", condition=c, clip=round(m, 3), clip_lo=ci[0], clip_hi=ci[1],
                            animal_ok=round(an, 3), attr_gain=round(ag, 3), steps=round(st, 2), total_s=round(ts, 3)))

# Pre-declared primary comparisons (Bonferroni: 5 comparisons -> 99% intervals)
PRIMARY = [("gnn_rag", "plain_matched"), ("gnn_rag_dpm", "plain_dpm_matched"), ("gnn_rag", "normal_rag"),
           ("gnn_rag", "plain_25"), ("gnn_rag", "abl_clip_planned")]
level = 1 - 0.05 / len(PRIMARY)
COMPARE = {}
print(f"\nPRIMARY comparisons ({level:.0%} intervals, Bonferroni-corrected)")
for a, b in PRIMARY:
    for metric in ("clip_score", "animal_ok", "total_s"):
        d, ci = cluster_ci(per_prompt_diff(a, b, metric), level=level)
        COMPARE[f"PRIMARY {a} vs {b} [{metric}]"] = (round(d, 3), ci)
        print(f"  {a:>12} vs {b:<18} {metric:>10}: Δ={d:+.3f}  CI {ci}{'  *' if ci[0] > 0 or ci[1] < 0 else ''}")
print("\nsecondary (95%, exploratory)")
for a, b in (("abl_gnn_fixed", "normal_rag"), ("gnn_rag", "turbo_4"), ("gnn_rag", "turbo_1")):
    if a in CONDS and b in CONDS:
        d, ci = cluster_ci(per_prompt_diff(a, b, "clip_score"))
        COMPARE[f"{a} vs {b} [clip_score]"] = (round(d, 3), ci)
        print(f"  {a:>14} vs {b:<12}: Δ={d:+.3f}  CI {ci}{'  *' if ci[0] > 0 or ci[1] < 0 else ''}")
for cat in ("color", "pattern", "part", "scene", "attr-only"):
    d, ci = cluster_ci(per_prompt_diff("gnn_rag", "plain_matched", "clip_score", cat))
    COMPARE[f"gnn_rag vs plain_matched [clip_score] ({cat})"] = (round(d, 3), ci)
    print(f"  gnn_rag vs plain_matched, {cat:>9}: Δ={d:+.3f}  CI {ci}{'  *' if ci[0] > 0 or ci[1] < 0 else ''}")

# Quality vs time (one point per condition, 95% CI on quality)
plt.figure(figsize=(7, 5))
for s in [x for x in SUMMARY if x["category"] == "all"]:
    plt.errorbar(s["total_s"], s["clip"], yerr=[[s["clip"] - s["clip_lo"]], [s["clip_hi"] - s["clip"]]],
                 fmt="o", capsize=3)
    plt.annotate(s["condition"], (s["total_s"], s["clip"]), textcoords="offset points", xytext=(5, 3), fontsize=8)
plt.xlabel("mean total seconds per image (T4)"); plt.ylabel("mean CLIP score (ViT-L/14)")
plt.title("Quality vs cost"); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig(os.path.join(OUT, "fig_quality_vs_time.png"), dpi=150); plt.show()

with open(os.path.join(OUT, "summary_table.csv"), "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(SUMMARY[0])); w.writeheader(); w.writerows(SUMMARY)
json.dump(dict(table_id=TABLE_ID, strength=STRENGTH, calibration=CAL,
               config=dict(SEED=SEED, N_SEEDS=N_SEEDS, N_PER_CATEGORY=N_PER_CATEGORY, N_NOMINAL=N_NOMINAL,
                           FIXED_STRENGTH=FIXED_STRENGTH, MP_BETA=MP_BETA, ALPHA_CLIP=ALPHA_CLIP,
                           SD=SD_ID, turbo=TURBO_ID if turbo is not None else None, judge=CLIP_JUDGE,
                           gpu=torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu",
                           torch=torch.__version__),
               prompts=PROMPTS, comparisons=COMPARE, summary=SUMMARY, rows=rows),
          open(os.path.join(OUT, f"results_{TABLE_ID}.json"), "w"), indent=2, default=str)
print("saved: summary_table.csv, fig_quality_vs_time.png, fig_calibration.png, results_*.json ->", OUT)

## Part B — Blind human evaluation
Raters see the request and two images (A/B, order randomised, method hidden) and answer
two questions. Share the printed link with 3–5 people; each enters a name and can resume.
Comparisons: GNN-RAG vs plain diffusion with matched steps, vs normal RAG, vs plain 25 steps.

### 15. Human-rating app

In [ ]:
import gradio as gr
HUMAN_PAIRS = [("gnn_rag", "plain_matched"), ("gnn_rag", "normal_rag"), ("gnn_rag", "plain_25")]
items_path = os.path.join(OUT, f"human_items_{TABLE_ID}.json")
ratings_path = os.path.join(OUT, f"human_ratings_{TABLE_ID}.jsonl")
if os.path.exists(items_path):
    ITEMS = json.load(open(items_path))
else:
    rs = random.Random(SEED + 77)
    pool = [(p["id"], k) for p in PROMPTS for k in range(N_SEEDS)]
    ITEMS = []
    for a, b in HUMAN_PAIRS:
        for pid, k in rs.sample(pool, min(HUMAN_N, len(pool))):
            left, right = (a, b) if rs.random() < 0.5 else (b, a)
            ITEMS.append(dict(prompt_id=pid, seed_k=k, prompt=PBY[pid]["text"], pair=f"{a}|{b}",
                              A=left, B=right, A_path=BY[(pid, k, left)]["path"], B_path=BY[(pid, k, right)]["path"]))
    rs.shuffle(ITEMS)
    for i, it in enumerate(ITEMS):
        it["item"] = i
    json.dump(ITEMS, open(items_path, "w"))

def rated_by(name):
    if not os.path.exists(ratings_path):
        return set()
    return {json.loads(l)["item"] for l in open(ratings_path) if json.loads(l)["rater"] == name}

def show_item(i):
    if i >= len(ITEMS):
        return "**All done — thank you!**", None, None
    it = ITEMS[i]
    return f"**Request:** {it['prompt']}  \n({i + 1} of {len(ITEMS)})", Image.open(it["A_path"]), Image.open(it["B_path"])

def next_unrated(name, start=0):
    done = rated_by(name)
    return next((i for i in range(start, len(ITEMS)) if i not in done), len(ITEMS))

def start(name):
    name = (name or "").strip()
    if not name:
        raise gr.Error("Enter your name first.")
    i = next_unrated(name)
    return (i, *show_item(i))

def submit(name, i, q1, q2):
    name = (name or "").strip()
    if not name or i is None:
        raise gr.Error("Enter your name and press Start.")
    if i >= len(ITEMS):
        return (i, *show_item(i), None, None)
    if not q1 or not q2:
        raise gr.Error("Please answer both questions.")
    with open(ratings_path, "a") as f:
        f.write(json.dumps(dict(rater=name, item=i, match=q1, quality=q2, time=time.time())) + "\n")
    j = next_unrated(name, i + 1)
    return (j, *show_item(j), None, None)

with gr.Blocks(title="Image rating") as rate_app:
    gr.Markdown("## Which image is better?\nYou will see a request and two images. Methods are hidden "
                "and the order is random. Judge only what you see.")
    with gr.Row():
        rater = gr.Textbox(label="Your name", scale=3); start_btn = gr.Button("Start / resume", scale=1)
    pos = gr.State(None); info = gr.Markdown()
    with gr.Row():
        imgA = gr.Image(label="A", type="pil"); imgB = gr.Image(label="B", type="pil")
    q_match = gr.Radio(["A", "B", "Equal"], label="Which image matches the request better?")
    q_qual = gr.Radio(["A", "B", "Equal"], label="Which image looks more natural / higher quality?")
    send = gr.Button("Submit and next", variant="primary")
    start_btn.click(start, [rater], [pos, info, imgA, imgB])
    send.click(submit, [rater, pos, q_match, q_qual], [pos, info, imgA, imgB, q_match, q_qual])
rate_app.queue().launch(share=True, debug=False)

### 16. Human-rating analysis (run after raters finish; can be rerun anytime)

In [ ]:
def human_results():
    if not os.path.exists(ratings_path):
        print("no ratings yet"); return
    R = [json.loads(l) for l in open(ratings_path)]
    print(f"{len(R)} ratings from {len({r['rater'] for r in R})} raters")
    for pair in [f"{a}|{b}" for a, b in HUMAN_PAIRS]:
        a, b = pair.split("|")
        for q in ("match", "quality"):
            by_prompt = defaultdict(list)
            for r in R:
                it = ITEMS[r["item"]]
                if it["pair"] != pair:
                    continue
                choice = r[q]
                win = 0.5 if choice == "Equal" else float(it[choice] == a)
                by_prompt[it["prompt_id"]].append(win)
            if by_prompt:
                m, ci = cluster_ci([np.mean(v) for v in by_prompt.values()])
                print(f"  {a} vs {b:<14} [{q:>7}] {a} preferred {m:.1%}  95% CI [{ci[0]:.1%}, {ci[1]:.1%}]"
                      f"  (50% = no difference; {len(by_prompt)} prompts)")
    agree, total_pairs = 0, 0
    by_item = defaultdict(list)
    for r in R:
        by_item[r["item"]].append(r["match"])
    for v in by_item.values():
        for i in range(len(v)):
            for j in range(i + 1, len(v)):
                total_pairs += 1; agree += v[i] == v[j]
    if total_pairs:
        print(f"  rater agreement on 'match': {agree / total_pairs:.1%} of {total_pairs} rater pairs")
human_results()

## Part C — Chatbot (uses the calibrated noise levels)

### 17. Chatbot

In [ ]:
_need = ["plan_gnn", "run_plain", "run_normal_rag", "run_gnn_rag", "judge", "MEM_EMB", "PIPES"]
_missing = [n for n in _need if n not in globals()]
assert not _missing, f"Not set up in this session: {_missing}. Run cells 2-9 first (and 11 for calibration)."
CALIBRATED = os.path.exists(CAL_PATH)
HISTORY = []
NAMES = ("Plain diffusion", "Normal RAG + diffusion", "GNN-RAG + diffusion")

def cost_md(name, r, score):
    return (f"**{name}**  \n{r['steps']} denoising steps · retrieval {r['retrieval_ms']:.1f} ms · "
            f"generation {r['gen_s']:.2f} s · **total {r['total_s']:.2f} s** · CLIP score {score:.1f}")

def plan_md(plan, normal_idx):
    pr = plan["parse"]
    miss = ", ".join(f"{a} ({c})" for c, a in plan["missing"]) or "nothing"
    missing_names = {a for _, a in plan["missing"]}
    coloured = {WORD2ATTR[pc.split()[-1]] for pc in pr.get("part_colors", [])}
    present = [a for a in pr["attrs"] if a not in missing_names and a not in coloured]
    return (f"**Understood:** animal = {pretty(pr['animal']) if pr['animal'] else '—'} · "
            f"attributes = {', '.join(pr['attrs']) or '—'}"
            f"{' · coloured parts = ' + ', '.join(pr['part_colors']) if pr.get('part_colors') else ''}"
            f" · unknown to the graph = {', '.join(pr['unknown']) or '—'}  \n"
            f"**Normal RAG** starts from: {pretty(CLASSES[MEM_CLASS[normal_idx]])} (fixed strength {FIXED_STRENGTH})  \n"
            f"**GNN-RAG** starts from: {pretty(plan['retrieved_animal'])} · must change: {miss}"
            f"{' · already in the start photo: ' + ', '.join(present) if present else ''}"
            f"{' · recoloured to ' + plan['recolor'] if plan['recolor'] else ''} · "
            f"strength {plan['strength']} → {plan['steps']}/{N_NOMINAL} steps "
            f"({'calibrated' if CALIBRATED else 'uncalibrated'} table)")

def summary_md():
    if not HISTORY:
        return ""
    lines = ["| pipeline | requests | mean steps | mean total s | mean CLIP score |", "|---|---|---|---|---|"]
    for name in NAMES:
        g = [h for h in HISTORY if h[1] == name]
        lines.append(f"| {name} | {len(g)} | {np.mean([h[3] for h in g]):.1f} | "
                     f"{np.mean([h[6] for h in g]):.2f} | {np.mean([h[7] for h in g]):.1f} |")
    return "\n".join(lines)

def chat(query, seed, sampler):
    query = (query or "").strip()
    if not query:
        raise gr.Error("Type what you want to see, e.g. 'a striped horse in the snow'.")
    seed, smp = int(seed or 0), ("dpm" if sampler.startswith("DPM") else "ddim")
    plan = plan_gnn(query)
    base = base_text(plan["parse"]["animal"]) if plan["parse"]["animal"] else "a photo of an animal"
    results = {NAMES[0]: run_plain(query, seed, sampler=smp),
               NAMES[1]: run_normal_rag(query, seed, sampler=smp),
               NAMES[2]: run_gnn_rag(query, seed, plan=plan, sampler=smp)}
    mds = []
    for name, r in results.items():
        score, _, _ = judge(r["img"], query, base)
        HISTORY.append([query, name, sampler, r["steps"], round(r["retrieval_ms"], 1), round(r["gen_s"], 2),
                        round(r["total_s"], 2), round(score, 1)])
        mds.append(cost_md(name, r, score))
    r0, r1, r2 = results.values()
    return (plan_md(plan, r1["idx"]), r0["img"], mds[0], r1["img"], r1["init"], mds[1],
            r2["img"], r2["init"], mds[2], [h[:] for h in HISTORY[::-1]], summary_md())

with gr.Blocks(title="GNN-RAG image diffusion") as demo:
    gr.Markdown("## GNN-RAG + diffusion vs normal RAG + diffusion vs plain diffusion\n"
                "Ask for an animal with any change: *a blue tiger*, *an elephant with blue horns*, "
                "*a tiger in the arctic*, *an animal with stripes and hooves*, *a five-legged dog*.")
    with gr.Row():
        q = gr.Textbox(label="Your request", placeholder="a striped horse in the snow", scale=5)
        seed_box = gr.Number(value=0, precision=0, label="seed", scale=1)
        smp_box = gr.Dropdown(["DDIM", "DPM-Solver++"], value="DDIM", label="sampler", scale=1)
        btn = gr.Button("Generate", variant="primary", scale=1)
    plan_box = gr.Markdown()
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Plain diffusion")
            img0 = gr.Image(label="result", type="pil"); md0 = gr.Markdown()
        with gr.Column():
            gr.Markdown("### Normal RAG + diffusion")
            img1 = gr.Image(label="result", type="pil"); init1 = gr.Image(label="retrieved start", type="pil", height=160)
            md1 = gr.Markdown()
        with gr.Column():
            gr.Markdown("### GNN-RAG + diffusion")
            img2 = gr.Image(label="result", type="pil"); init2 = gr.Image(label="retrieved start (after edit)", type="pil", height=160)
            md2 = gr.Markdown()
    gr.Markdown("### Session averages")
    summary_box = gr.Markdown()
    hist = gr.Dataframe(headers=["request", "pipeline", "sampler", "steps", "retrieval ms", "generation s",
                                 "total s", "CLIP score"], interactive=False)
    outs = [plan_box, img0, md0, img1, init1, md1, img2, init2, md2, hist, summary_box]
    btn.click(chat, [q, seed_box, smp_box], outs)
    q.submit(chat, [q, seed_box, smp_box], outs)
demo.queue().launch(share=True, debug=False)

## Scope and limitations

The experiments use AwA2 (50 animal classes and 85 attributes), Stable Diffusion 1.5, a rule-based prompt parser, and CLIP-based automatic scoring. Message passing is training-free. Recolouring changes the full image, and exact counting edits are not supported. Calibration uses eight prompts per edit type; runs are configured for a Colab T4 with fp16.

A broader evaluation would add another attribute dataset, a learned graph model, and a larger human study.
